# 04 · Results: performance, reliability, robustness and model selection

**Brief sections 11 and 12.** Everything here is computed from the outer-fold predictions of notebook 03. They are Downtown development boreholes, each predicted by a model that never saw it, its fold, or its labels during tuning.

Reported for every model × scenario × block size:

* **Metrics:** accuracy, balanced accuracy, Macro-F1, Clay and Sand recall, MCC, ROC-AUC, PR-AUC (Clay and Sand as positive), Brier score, confusion matrices;
* **Fold-level:** each outer fold individually, then mean ± SD and minimum over folds;
* **Pooled:** pooled out-of-fold metrics, labelled separately because they mix models with different calibration;
* **Borehole-level:** accuracy for every test borehole; Macro-F1 and balanced accuracy only where a borehole has both classes;
* **Depth bands:** 0–5, 5–10 and 10–15 m;
* **Distance sensitivity:** distance from a test borehole to its nearest training borehole;
* **Confidence and uncertainty:** reliability, entropy;
* **Robustness:** 100 / 200 / 300 m blocks, spatial purge, Scenario A vs B under identical test folds;
* **Profiles and error maps:** actual vs predicted columns and spatial error maps (restricted outputs: they show borehole codes and locations).

**Model selection** follows the rule fixed before modelling (design report §5.7), applied to the primary block size without purge.

1. A candidate is eligible only if its mean Clay recall and mean Sand recall are both ≥ 0.50, and no fold has a class recall below 0.25.
2. Eligible candidates are ranked by mean outer-fold Macro-F1.
3. Within one standard error of the best, the candidate with the lowest fold SD wins, then the lowest Brier score.

Choosing the best of ten candidates by these scores is itself optimistic, so the external holdout (notebook 05) is the independent check. Poor folds and poor classes are reported, not hidden.

In [ ]:
import os, sys, json
from pathlib import Path

# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner())

from siemreap import metrics as M, plots, geo
AGG = config.stage_dir(cfg, "04_results", "aggregate")
RES = config.stage_dir(cfg, "04_results", "restricted")
preds = pd.read_csv(config.stage_dir(cfg, "03_nested_cv", "restricted") / "outer_predictions.csv")
run_info = pd.read_csv(config.stage_dir(cfg, "03_nested_cv", "aggregate") / "run_info.csv")
primary = cfg["smoke"]["block_sizes_m"][0] if MODE == "smoke" else cfg["blockcv"]["primary_block_m"]
print(len(preds), "outer-fold predictions;", preds.groupby(M.RUN_KEYS).ngroups, "runs; primary block", primary, "m")

## 1. Outer-fold metrics, mean ± SD, and pooled out-of-fold metrics

In [ ]:
folds_t = M.fold_table(preds)
summary = M.fold_summary(folds_t)
pooled = M.pooled_table(preds)
folds_t.to_csv(AGG / "metrics_by_outer_fold.csv", index=False)
summary.to_csv(AGG / "metrics_fold_mean_sd.csv", index=False)
pooled.to_csv(AGG / "metrics_pooled_oof.csv", index=False)
show = ["scenario", "block_m", "purge_m", "model", "macro_f1_mean", "macro_f1_sd", "macro_f1_min", "balanced_accuracy_mean",
        "recall_clay_mean", "recall_sand_mean", "mcc_mean", "roc_auc_mean", "brier_mean", "n_folds"]
display(summary[(summary.purge_m == 0)][show].round(3).sort_values(["block_m", "scenario", "macro_f1_mean"], ascending=[True, True, False]))

In [ ]:
display(pooled[pooled.purge_m == 0][M.RUN_KEYS + M.METRICS + ["tn", "fp", "fn", "tp"]].round(3))

In [ ]:
main = folds_t[(folds_t.block_m == primary) & (folds_t.purge_m == 0)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4), dpi=150)
plots.fold_dotplot(main, "macro_f1", f"Macro-F1 by outer fold, {primary} m blocks", ax=axes[0])
plots.fold_dotplot(main, "balanced_accuracy", f"Balanced accuracy by outer fold, {primary} m blocks", ax=axes[1])
fig.tight_layout(); fig.savefig(AGG / "fig_outer_fold_metrics.png", bbox_inches="tight")

## 2. Training extent: Scenario A vs B on identical test folds

In [ ]:
w = folds_t[folds_t.purge_m == 0].pivot_table(index=["block_m", "model", "outer_fold"], columns="scenario",
                                              values=["macro_f1", "recall_clay", "recall_sand"])
delta = pd.DataFrame({m: w[(m, "B")] - w[(m, "A")] for m in ["macro_f1", "recall_clay", "recall_sand"]}) if "B" in folds_t.scenario.unique() else None
if delta is not None:
    d = delta.groupby(["block_m", "model"]).agg(["mean", "std", "min", "max"]).round(3)
    display(d); d.to_csv(AGG / "scenario_B_minus_A.csv")

## 3. Depth bands, distance sensitivity, reliability

In [ ]:
depth_t = M.depth_band_table(preds, cfg["depth"]["bands_m"]); depth_t.to_csv(AGG / "metrics_by_depth_band.csv", index=False)
dist_t = M.distance_table(preds); dist_t.to_csv(AGG / "metrics_by_distance_to_training.csv", index=False)
rel = M.reliability_table(preds); rel.to_csv(AGG / "reliability_pooled.csv", index=False)
sel_runs = lambda t: t[(t.block_m == primary) & (t.purge_m == 0)]
axes = plots.band_plot(sel_runs(depth_t), "depth_band", "macro_f1", "Macro-F1 by depth band")
axes[0].figure.savefig(AGG / "fig_macro_f1_by_depth_band.png", bbox_inches="tight")
axes = plots.band_plot(sel_runs(dist_t), "distance_bin", "macro_f1", "Macro-F1 by distance to nearest training borehole")
axes[0].figure.savefig(AGG / "fig_macro_f1_by_distance.png", bbox_inches="tight")
display(sel_runs(dist_t)[M.RUN_KEYS + ["distance_bin", "n", "macro_f1", "balanced_accuracy"]].round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.6), dpi=150)
plots.reliability_plot(sel_runs(rel), ax=ax); fig.savefig(AGG / "fig_reliability.png", bbox_inches="tight")
unc = preds.assign(entropy=M.entropy(preds.p_clay), correct=lambda d: d.y == d.y_pred)
u = sel_runs(unc).groupby(["scenario", "model", "correct"]).entropy.mean().unstack().round(3)
display(u.rename(columns={True: "entropy_correct", False: "entropy_wrong"}))

## 4. Robustness: block size and spatial purge

In [ ]:
rob = summary[summary.purge_m == 0].pivot_table(index=["scenario", "model"], columns="block_m",
                                               values="macro_f1_mean").round(3)
display(rob); rob.to_csv(AGG / "macro_f1_by_block_size.csv")
pur = summary[summary.block_m == primary].pivot_table(index=["scenario", "model"], columns="purge_m",
                                                      values="macro_f1_mean").round(3)
display(pur); pur.to_csv(AGG / "macro_f1_by_purge_distance.csv")
if "gw_fallback_rate" in run_info:
    display(run_info.dropna(subset=["gw_fallback_rate"]).groupby(["block_m", "scenario", "model"]).gw_fallback_rate.mean())

## 5. Borehole-level reliability, profiles and error maps (restricted)

In [ ]:
bh_t = M.borehole_table(preds); bh_t.to_csv(RES / "metrics_by_borehole.csv", index=False)
agg_bh = (bh_t.groupby(M.RUN_KEYS).agg(boreholes=("BH_Code", "size"), accuracy_median=("accuracy", "median"),
          accuracy_p10=("accuracy", lambda s: s.quantile(0.1)), macro_f1_median=("macro_f1", "median"),
          share_accuracy_below_0_5=("accuracy", lambda s: (s < 0.5).mean())).round(3).reset_index())
agg_bh.to_csv(AGG / "borehole_level_summary.csv", index=False); display(agg_bh[agg_bh.purge_m == 0])

In [ ]:
best_run = summary[(summary.block_m == primary) & (summary.purge_m == 0) & (summary.model != "DEPTH_PRIOR")] \
    .sort_values("macro_f1_mean", ascending=False).iloc[0]
pr = preds[(preds.model == best_run.model) & (preds.scenario == best_run.scenario) & (preds.block_m == primary) & (preds.purge_m == 0)]
codes = pr.drop_duplicates("BH_Code").sort_values(["outer_fold", "dist_nearest_train_m"]).BH_Code.head(24)
fig = plots.profiles(pr, codes, title=f"Observed vs predicted, {best_run.model} ({best_run.scenario}), {primary} m blocks")
fig.savefig(RES / "fig_profiles_example.png", bbox_inches="tight")
poly = geo.load_polygon(cfg, "downtown_shp")
xy = preds.drop_duplicates("BH_Code").set_index("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]]
b = bh_t[(bh_t.model == best_run.model) & (bh_t.scenario == best_run.scenario) & (bh_t.block_m == primary) & (bh_t.purge_m == 0)]
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
plots.error_map(b, xy, poly, "accuracy", f"Borehole accuracy, {best_run.model} ({best_run.scenario})", ax=ax)
fig.savefig(RES / "fig_error_map.png", bbox_inches="tight")

## 6. Pre-declared model selection

In [ ]:
sel = M.select_model(summary, {**cfg, "blockcv": {**cfg["blockcv"], "primary_block_m": primary}})
display(sel["table"][["scenario", "model", "eligible", "macro_f1_mean", "macro_f1_sd", "recall_clay_mean", "recall_sand_mean",
                      "recall_clay_min", "recall_sand_min", "brier_mean"]].round(3).sort_values("macro_f1_mean", ascending=False))
record = {k: v for k, v in sel.items() if k != "table"}
record["mode"] = MODE
(AGG / "selected_model.json").write_text(json.dumps(record, indent=1, default=str))
print(json.dumps(record, indent=1, default=str))

**Interpretation checklist (fill in for the thesis):**

* Does any model beat the depth-band prior, and by how much? This measures what the horizontal coordinates add.
* Where does performance drop: which folds, depth bands and distance classes? Is that geologically plausible (e.g. the clayey-sand transition at 8–12 m)?
* Does Scenario B help or hurt Downtown predictions, and in which depth band? (The 5 km ring is sandier at mid-depth.)
* How sensitive are the conclusions to the block size (100 vs 200 vs 300 m) and to the purge?
* Remember that the Clay/Sand boundary sits at 50 % fines and about a quarter of tested samples lie within 45–55 %, so some error is irreducible.